# Meta-Learner: Train & Evaluate Posterior Stacking

This notebook trains a meta-learner that combines posteriors from multiple
independently trained SBI models into a single, better-calibrated posterior.

**Scope:** Model combination only. OOD detection lives in `ood_tests/`.

**Steps:**
1. Load N frozen base models
2. Extract posterior summaries on validation data
3. Train the meta-MLP
4. Evaluate calibration on held-out ID data
5. Export the trained model

In [ ]:
import sys
from pathlib import Path
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt

project_root = Path("../../..").resolve()
sys.path.insert(0, str(project_root))

from sbi4atmret.config.configs import BaseConfig
from sbi4atmret.models.ModelBase import BaseModel
from sbi4atmret.models.meta_learner import (
    MetaMLP,
    load_base_models,
    extract_validation_summaries,
    train_meta_learner,
    meta_predict,
)

## 1. Configuration

In [ ]:
config_path = project_root / "experiments/config_MiriGeminiHST_cloudfree.yaml"

with open(config_path) as f:
    config_dict = yaml.safe_load(f)

config = BaseConfig(**config_dict)
n_params = len(config.prior.parameters)
print(f"Number of parameters: {n_params}")

device = "cuda" if torch.cuda.is_available() else "cpu"

# Output directory for the trained meta-learner
meta_save_dir = Path("trained_meta_learner")
meta_save_dir.mkdir(exist_ok=True)

## 2. Load Frozen Base Models

In [ ]:
checkpoint_paths = [
    Path("path/to/model_seed1/states_800.pth"),
    Path("path/to/model_seed2/states_800.pth"),
    Path("path/to/model_seed3/states_800.pth"),
]

base_models = load_base_models(
    checkpoint_paths,
    model_builder=lambda: BaseModel(config).build(),
    device=device,
)

print(f"Loaded {len(base_models)} base models")

## 3. Extract Summaries on Validation Set

In [ ]:
# Set up validation data
# from sbi4atmret.datasets.DatasetBase import Dataset
# from sbi4atmret.runtime.batch_processor import BatchProcessor
# dataset = Dataset(config)
# val_loader = ...  
# batch_processor = BatchProcessor(pipe=..., noise=..., device=device)

include_embeddings = True
n_samples = 256

summaries, thetas = extract_validation_summaries(
    base_models,
    dataloader=val_loader,
    batch_processor=batch_processor,
    n_samples=n_samples,
    include_embeddings=include_embeddings,
    device=device,
)

print(f"Summaries: {summaries.shape}")
print(f"Thetas: {thetas.shape}")

## 4. Train Meta-Learner

In [ ]:
summary_dim = summaries.shape[-1]

meta_model = MetaMLP(
    input_dim=summary_dim,
    n_params=n_params,
    hidden_dims=[256, 128],
    dropout=0.1,
)

print(f"Input dim: {summary_dim}")
print(f"Output dim: {2 * n_params}")
print(f"Parameters: {sum(p.numel() for p in meta_model.parameters()):,}")

In [ ]:
save_path = meta_save_dir / "meta_learner_best.pth"

history = train_meta_learner(
    meta_model,
    summaries,
    thetas,
    n_epochs=200,
    batch_size=512,
    lr=1e-3,
    weight_decay=1e-4,
    val_fraction=0.1,
    device=device,
    save_path=save_path,
)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history["train_loss"], label="Train")
ax.plot(history["val_loss"], label="Validation")
ax.set_xlabel("Epoch")
ax.set_ylabel("Gaussian NLL Loss")
ax.set_title("Meta-Learner Training")
ax.legend()
plt.tight_layout()
plt.savefig(meta_save_dir / "training_history.pdf", bbox_inches="tight")
plt.show()

## 5. Evaluate Calibration on In-Distribution Data

Check that the meta-learner's predicted means and stds are well-calibrated.

In [ ]:
meta_model.eval()

with torch.no_grad():
    val_mean, val_std = meta_model.predict(summaries.to(device))
    val_mean, val_std = val_mean.cpu(), val_std.cpu()

# Z-scores: if calibrated, these should be ~ N(0,1)
z_scores = ((thetas - val_mean) / val_std).numpy()

print(f"Z-score mean: {z_scores.mean():.3f} (expect ~0)")
print(f"Z-score std: {z_scores.std():.3f} (expect ~1)")

In [ ]:
from scipy.stats import norm

# Coverage plot: fraction within k-sigma credible interval
alphas = np.linspace(0, 1, 100)
coverage = []
for alpha in alphas:
    z_crit = norm.ppf(0.5 + alpha / 2)
    frac_within = (np.abs(z_scores) < z_crit).mean()
    coverage.append(frac_within)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(alphas, coverage, color="steelblue", label="Meta-learner")
ax.plot([0, 1], [0, 1], "k--", label="Ideal")
ax.set_xlabel(r"Credibility level $1-\alpha$")
ax.set_ylabel("Coverage probability")
ax.set_title("Meta-Learner Calibration (ID)")
ax.legend()
plt.tight_layout()
plt.savefig(meta_save_dir / "calibration_ID.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# Per-parameter residual analysis
fig, axes = plt.subplots(4, 7, figsize=(16, 10))
for i, ax in enumerate(axes.flat[:n_params]):
    ax.hist(z_scores[:, i], bins=30, density=True, alpha=0.7, color="steelblue")
    x_grid = np.linspace(-4, 4, 100)
    ax.plot(x_grid, norm.pdf(x_grid), "k--", linewidth=0.8)
    ax.set_title(f"p{i}", fontsize=8)
    ax.set_xlim(-4, 4)
for ax in axes.flat[n_params:]:
    ax.set_visible(False)
fig.suptitle("Per-parameter z-score distributions (should be ~N(0,1))")
plt.tight_layout()
plt.savefig(meta_save_dir / "per_param_calibration.pdf", bbox_inches="tight")
plt.show()

## 6. Inference on Real Observation

In [ ]:
# x_obs = torch.from_numpy(observation.full_observation).unsqueeze(0).float()

mean, std = meta_predict(
    base_models,
    meta_model,
    x_obs,
    n_samples=1024,
    include_embeddings=include_embeddings,
    device=device,
)

print("Meta-learner posterior estimate:")
for i in range(n_params):
    print(f"  param {i:2d}: {mean[0, i].item():.4f} +/- {std[0, i].item():.4f}")

In [ ]:
# Compare individual models vs meta-learner
individual_means = []
individual_stds = []

with torch.no_grad():
    for model in base_models:
        x_emb = model.embedding(x_obs.to(device))
        posterior = model.flow.flow(x_emb)
        samples = posterior.sample((2048,))
        individual_means.append(samples.mean(dim=0).cpu())
        individual_stds.append(samples.std(dim=0).cpu())

n_show = min(8, n_params)
fig, axes = plt.subplots(2, 4, figsize=(14, 6))

for i, ax in enumerate(axes.flat[:n_show]):
    for j, (m, s) in enumerate(zip(individual_means, individual_stds)):
        ax.errorbar(j, m[0, i].item(), yerr=s[0, i].item(),
                    fmt='o', alpha=0.5, capsize=3)
    ax.errorbar(len(base_models), mean[0, i].item(), yerr=std[0, i].item(),
                fmt='s', color='red', markersize=8, capsize=3)
    ax.set_title(f"Param {i}")
    ticks = [f"M{j+1}" for j in range(len(base_models))] + ["Meta"]
    ax.set_xticks(range(len(ticks)))
    ax.set_xticklabels(ticks, fontsize=7)

fig.suptitle("Individual Models vs Meta-Learner")
plt.tight_layout()
plt.savefig(meta_save_dir / "models_vs_meta.pdf", bbox_inches="tight")
plt.show()

## 7. Export Trained Model

In [ ]:
# Save full config for reproducibility
export = {
    "model_state_dict": meta_model.state_dict(),
    "input_dim": summary_dim,
    "n_params": n_params,
    "hidden_dims": [256, 128],
    "include_embeddings": include_embeddings,
    "n_base_models": len(base_models),
    "checkpoint_paths": [str(p) for p in checkpoint_paths],
    "history": history,
}

torch.save(export, meta_save_dir / "meta_learner_export.pt")
print(f"Exported to {meta_save_dir / 'meta_learner_export.pt'}")